# SBI Dataset-size Diagnostics

Analyze the cluster SBI sweep outputs as a function of the number of training datapoints. The notebook discovers every `N*` run under `HPC_output/HalfDome/Emulator/Dataset_test`, then plots posterior width normalized by the prior range, normalized posterior variance, pull, and loss summaries versus `n_train`.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PARAM_NAMES = [
    "P0",
    "xc",
    "beta",
    "alpha_m_P0",
    "alpha_m_xc",
    "alpha_m_beta",
    "alpha_z_P0",
    "alpha_z_xc",
    "alpha_z_beta",
]

# Fiducial values used by the existing cluster SBI analysis notebook.
THETA_TRUE = np.array([18.1, 0.497, 4.35, 0.154, -0.00865, 0.0393, -0.758, 0.731, 0.415], dtype=float)

DEFAULT_PRIOR = {
    "P0": [2.0, 25.0],
    "xc": [0.12, 0.70],
    "beta": [3.8, 5.2],
    "alpha_m_P0": [0.0, 0.30],
    "alpha_m_xc": [-0.08, 0.08],
    "alpha_m_beta": [0.0, 0.08],
    "alpha_z_P0": [-1.10, -0.40],
    "alpha_z_xc": [0.10, 0.90],
    "alpha_z_beta": [0.25, 0.55],
}

PULL_KIND = "mean"  # "mean" or "median"
XSCALE = "log"
SAVE_FIGURES = True
SAVE_TABLES = True
FIGURE_DPI = 160


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "HPC_output").exists() and (candidate / "SBI_analysis").exists():
            return candidate
    fallback = Path("/home/cbllover/HalfDome")
    if fallback.exists():
        return fallback
    return start


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "HPC_output" / "HalfDome" / "Emulator" / "Dataset_test"
if not DATA_ROOT.exists():
    DATA_ROOT = Path(r"\\wsl.localhost\myrootfs\home\cbllover\HalfDome\HPC_output\HalfDome\Emulator\Dataset_test")

CONFIG_PATH = REPO_ROOT / "SBI_analysis" / "configs" / "default_cluster.json"
FIGURE_DIR = DATA_ROOT / "diagnostic_plots"
TABLE_DIR = DATA_ROOT / "diagnostic_tables"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

print(f"Repo root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Config path: {CONFIG_PATH}")

In [ ]:
def read_json(path: Path) -> dict:
    if not path.exists() or path.stat().st_size == 0:
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_prior_bounds(config_path: Path, param_names: list[str]) -> tuple[np.ndarray, np.ndarray]:
    prior = dict(DEFAULT_PRIOR)
    if config_path.exists():
        config = read_json(config_path)
        prior.update(config.get("prior", {}))
    missing = [name for name in param_names if name not in prior]
    if missing:
        raise KeyError(f"Missing prior bounds for: {missing}")
    low = np.array([prior[name][0] for name in param_names], dtype=float)
    high = np.array([prior[name][1] for name in param_names], dtype=float)
    if np.any(high <= low):
        raise ValueError("Every prior upper bound must be greater than the lower bound")
    return low, high


def n_train_from_dir(run_dir: Path, metadata: dict) -> int:
    if "n_train" in metadata:
        return int(metadata["n_train"])
    match = re.fullmatch(r"N(\d+)", run_dir.name)
    if not match:
        raise ValueError(f"Cannot infer n_train from {run_dir}")
    return int(match.group(1))


def discover_runs(data_root: Path) -> list[dict]:
    runs = []
    for run_dir in sorted(data_root.rglob("N*")):
        if not run_dir.is_dir() or not re.fullmatch(r"N\d+", run_dir.name):
            continue
        metadata_path = run_dir / "run_metadata.json"
        metadata = read_json(metadata_path)
        posterior_samples_path = run_dir / "posterior_samples.npy"
        loss_history_path = run_dir / "loss_history.npz"
        if not posterior_samples_path.exists() and not loss_history_path.exists():
            continue
        runs.append({
            "n_train": n_train_from_dir(run_dir, metadata),
            "job_id": run_dir.parent.parent.name if run_dir.parent.parent != data_root else "",
            "group": run_dir.parent.name,
            "run_dir": run_dir,
            "metadata_path": metadata_path,
            "posterior_samples_path": posterior_samples_path,
            "loss_history_path": loss_history_path,
            "metadata": metadata,
            "posterior_available": posterior_samples_path.exists() and posterior_samples_path.stat().st_size > 0,
            "loss_available": loss_history_path.exists() and loss_history_path.stat().st_size > 0,
        })
    if not runs:
        raise FileNotFoundError(f"No N* runs found under {data_root}")
    return sorted(runs, key=lambda row: (row["n_train"], str(row["run_dir"])))


def format_value(value) -> str:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (bool, np.bool_)):
        return "yes" if value else "no"
    try:
        value_float = float(value)
    except (TypeError, ValueError):
        return str(value)
    if not np.isfinite(value_float):
        return "nan"
    if value_float.is_integer():
        return str(int(value_float))
    if abs(value_float) >= 1000 or (0 < abs(value_float) < 1e-3):
        return f"{value_float:.4g}"
    return f"{value_float:.5f}"


def print_rows(rows: list[dict], columns: list[str], max_rows: int | None = None) -> None:
    shown = rows if max_rows is None else rows[:max_rows]
    widths = {col: max(len(col), *(len(format_value(row.get(col, ""))) for row in shown)) for col in columns}
    header = "  ".join(col.ljust(widths[col]) for col in columns)
    print(header)
    print("  ".join("-" * widths[col] for col in columns))
    for row in shown:
        print("  ".join(format_value(row.get(col, "")).ljust(widths[col]) for col in columns))
    if max_rows is not None and len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")


prior_low, prior_high = load_prior_bounds(CONFIG_PATH, PARAM_NAMES)
prior_range = prior_high - prior_low
runs = discover_runs(DATA_ROOT)

print(f"Discovered {len(runs)} run directories")
print_rows(
    runs,
    ["n_train", "job_id", "group", "posterior_available", "loss_available"],
)

In [ ]:
def posterior_stats(samples_path: Path) -> dict[str, np.ndarray | tuple[int, ...]]:
    samples = np.load(samples_path, mmap_mode="r")
    if samples.ndim != 2:
        raise ValueError(f"Expected posterior samples with shape (n_samples, n_params), got {samples.shape}")
    if samples.shape[1] != len(PARAM_NAMES):
        raise ValueError(f"Expected {len(PARAM_NAMES)} parameters, got {samples.shape[1]} in {samples_path}")
    return {
        "shape": tuple(samples.shape),
        "mean": np.asarray(np.mean(samples, axis=0), dtype=float),
        "median": np.asarray(np.median(samples, axis=0), dtype=float),
        "std": np.asarray(np.std(samples, axis=0, ddof=1), dtype=float),
        "variance": np.asarray(np.var(samples, axis=0, ddof=1), dtype=float),
        "q16": np.asarray(np.percentile(samples, 16, axis=0), dtype=float),
        "q84": np.asarray(np.percentile(samples, 84, axis=0), dtype=float),
        "q2.5": np.asarray(np.percentile(samples, 2.5, axis=0), dtype=float),
        "q97.5": np.asarray(np.percentile(samples, 97.5, axis=0), dtype=float),
    }


def loss_stats(loss_path: Path) -> dict[str, float | int]:
    out = {
        "epochs_training": np.nan,
        "epochs_validation": np.nan,
        "first_training_loss": np.nan,
        "final_training_loss": np.nan,
        "first_validation_loss": np.nan,
        "final_validation_loss": np.nan,
        "best_validation_loss": np.nan,
        "final_loss_gap": np.nan,
        "training_loss_drop": np.nan,
        "validation_loss_drop": np.nan,
    }
    if not loss_path.exists() or loss_path.stat().st_size == 0:
        return out
    with np.load(loss_path) as loss_file:
        train_logp = np.asarray(loss_file.get("training_log_probs", []), dtype=float)
        val_logp = np.asarray(loss_file.get("validation_log_probs", []), dtype=float)
        best_logp = np.asarray(loss_file.get("best_validation_log_prob", []), dtype=float)

    if train_logp.size:
        out["epochs_training"] = int(train_logp.size)
        out["first_training_loss"] = float(-train_logp[0])
        out["final_training_loss"] = float(-train_logp[-1])
        out["training_loss_drop"] = float(train_logp[-1] - train_logp[0])
    if val_logp.size:
        out["epochs_validation"] = int(val_logp.size)
        out["first_validation_loss"] = float(-val_logp[0])
        out["final_validation_loss"] = float(-val_logp[-1])
        out["validation_loss_drop"] = float(val_logp[-1] - val_logp[0])
        out["best_validation_loss"] = float(-np.nanmax(val_logp))
    if best_logp.size:
        out["best_validation_loss"] = float(-np.nanmax(best_logp))
    if np.isfinite(out["final_validation_loss"]) and np.isfinite(out["final_training_loss"]):
        out["final_loss_gap"] = float(out["final_validation_loss"] - out["final_training_loss"])
    return out


posterior_rows = []
run_rows = []

for run in runs:
    row = {
        "n_train": run["n_train"],
        "job_id": run["job_id"],
        "group": run["group"],
        "run_dir": run["run_dir"],
        "posterior_available": run["posterior_available"],
        "loss_available": run["loss_available"],
    }
    row.update(loss_stats(run["loss_history_path"]))

    if run["posterior_available"]:
        stats = posterior_stats(run["posterior_samples_path"])
        row["posterior_sample_count"] = int(stats["shape"][0])
        for index, name in enumerate(PARAM_NAMES):
            std = float(stats["std"][index])
            variance = float(stats["variance"][index])
            mean = float(stats["mean"][index])
            median = float(stats["median"][index])
            true = float(THETA_TRUE[index])
            posterior_rows.append({
                "n_train": run["n_train"],
                "job_id": run["job_id"],
                "group": run["group"],
                "parameter": name,
                "true": true,
                "prior_low": float(prior_low[index]),
                "prior_high": float(prior_high[index]),
                "prior_range": float(prior_range[index]),
                "mean": mean,
                "median": median,
                "std": std,
                "variance": variance,
                "std_over_prior_range": float(std / prior_range[index]),
                "variance_over_prior_range2": float(variance / (prior_range[index] ** 2)),
                "q16": float(stats["q16"][index]),
                "q84": float(stats["q84"][index]),
                "q2.5": float(stats["q2.5"][index]),
                "q97.5": float(stats["q97.5"][index]),
                "pull_mean": float((mean - true) / std) if std > 0 else np.nan,
                "pull_median": float((median - true) / std) if std > 0 else np.nan,
            })
    else:
        row["posterior_sample_count"] = 0
    run_rows.append(row)

run_rows = sorted(run_rows, key=lambda row: row["n_train"])
posterior_rows = sorted(posterior_rows, key=lambda row: (row["parameter"], row["n_train"]))

print("Run-level summary")
print_rows(
    run_rows,
    ["n_train", "job_id", "group", "posterior_sample_count", "loss_available", "final_validation_loss", "best_validation_loss", "final_loss_gap"],
)

print("\nPosterior summary preview")
print_rows(
    posterior_rows,
    ["n_train", "parameter", "std_over_prior_range", "variance_over_prior_range2", "pull_mean", "pull_median"],
    max_rows=18,
)

In [ ]:
def write_csv_table(path: Path, rows: list[dict], columns: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


run_columns = [
    "n_train", "job_id", "group", "run_dir", "posterior_available", "posterior_sample_count", "loss_available",
    "epochs_training", "epochs_validation", "first_training_loss", "final_training_loss",
    "first_validation_loss", "final_validation_loss", "best_validation_loss", "final_loss_gap",
    "training_loss_drop", "validation_loss_drop",
]
posterior_columns = [
    "n_train", "job_id", "group", "parameter", "true", "prior_low", "prior_high", "prior_range",
    "mean", "median", "std", "variance", "std_over_prior_range", "variance_over_prior_range2",
    "q16", "q84", "q2.5", "q97.5", "pull_mean", "pull_median",
]

if SAVE_TABLES:
    write_csv_table(TABLE_DIR / "run_loss_summary.csv", run_rows, run_columns)
    write_csv_table(TABLE_DIR / "posterior_parameter_summary.csv", posterior_rows, posterior_columns)
    print(f"Saved CSV summaries to {TABLE_DIR}")

In [ ]:
def save_figure(fig, filename: str) -> None:
    if not SAVE_FIGURES:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"Saved {path}")


def values_for_parameter(metric: str, parameter: str) -> tuple[np.ndarray, np.ndarray]:
    rows = [row for row in posterior_rows if row["parameter"] == parameter and np.isfinite(row.get(metric, np.nan))]
    rows = sorted(rows, key=lambda row: row["n_train"])
    x = np.array([row["n_train"] for row in rows], dtype=float)
    y = np.array([row[metric] for row in rows], dtype=float)
    return x, y


def plot_metric_facets(metric: str, ylabel: str, title: str, filename: str, hlines: list[float] | None = None):
    ncols = 3
    nrows = math.ceil(len(PARAM_NAMES) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.05 * nrows), sharex=True)
    axes = np.asarray(axes).ravel()
    for ax, parameter in zip(axes, PARAM_NAMES):
        x, y = values_for_parameter(metric, parameter)
        ax.plot(x, y, marker="o", linewidth=1.8, markersize=4.5)
        if hlines:
            for hline in hlines:
                style = "-" if hline == 0 else "--"
                alpha = 0.7 if hline == 0 else 0.35
                ax.axhline(hline, color="black", linestyle=style, linewidth=0.9, alpha=alpha)
        ax.set_title(parameter)
        ax.set_xscale(XSCALE)
        ax.grid(True, which="both", alpha=0.25)
        ax.set_xlabel("number of datapoints")
        ax.set_ylabel(ylabel)
    for ax in axes[len(PARAM_NAMES):]:
        ax.axis("off")
    fig.suptitle(title, y=1.01, fontsize=14)
    fig.tight_layout()
    save_figure(fig, filename)
    return fig


def plot_metric_overlay(metric: str, ylabel: str, title: str, filename: str):
    fig, ax = plt.subplots(figsize=(9.5, 5.6))
    for parameter in PARAM_NAMES:
        x, y = values_for_parameter(metric, parameter)
        ax.plot(x, y, marker="o", linewidth=1.7, markersize=4, label=parameter)
    ax.set_xscale(XSCALE)
    ax.set_xlabel("number of datapoints")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(ncol=3, fontsize=8, frameon=False)
    fig.tight_layout()
    save_figure(fig, filename)
    return fig

## Posterior Width and Variance

`std / prior_range` is the most directly interpretable posterior width fraction. `variance / prior_range^2` is the dimensionless normalized variance.

In [ ]:
plot_metric_overlay(
    "std_over_prior_range",
    "posterior std / prior range",
    "Posterior width vs number of datapoints",
    "posterior_std_over_prior_range_overlay.png",
);

plot_metric_facets(
    "std_over_prior_range",
    "std / prior range",
    "Posterior width by parameter",
    "posterior_std_over_prior_range_facets.png",
);

plot_metric_facets(
    "variance_over_prior_range2",
    "variance / prior range$^2$",
    "Normalized posterior variance by parameter",
    "posterior_variance_over_prior_range2_facets.png",
);

## Pull

The pull is `(posterior center - true value) / posterior std`. Use `PULL_KIND = "mean"` or `"median"` in the configuration cell to choose the posterior center.

In [ ]:
if PULL_KIND not in {"mean", "median"}:
    raise ValueError('PULL_KIND must be "mean" or "median"')

pull_metric = f"pull_{PULL_KIND}"
plot_metric_facets(
    pull_metric,
    f"{PULL_KIND} pull",
    f"Posterior {PULL_KIND} pull by parameter",
    f"posterior_{PULL_KIND}_pull_facets.png",
    hlines=[-1, 0, 1],
);

## Loss Variation

The training artifacts store log probabilities. The plots below use loss = `-log_prob`, so lower is better. Runs without `loss_history.npz` are skipped in the loss plots but remain in posterior plots.

In [ ]:
def finite_loss_rows() -> list[dict]:
    rows = [row for row in run_rows if row["loss_available"] and np.isfinite(row["final_validation_loss"])]
    return sorted(rows, key=lambda row: row["n_train"])


def plot_loss_summary():
    rows = finite_loss_rows()
    if not rows:
        print("No loss histories found.")
        return None
    x = np.array([row["n_train"] for row in rows], dtype=float)
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for key, label in [
        ("final_training_loss", "final training loss"),
        ("final_validation_loss", "final validation loss"),
        ("best_validation_loss", "best validation loss"),
    ]:
        y = np.array([row[key] for row in rows], dtype=float)
        ax.plot(x, y, marker="o", linewidth=1.9, markersize=5, label=label)
    ax.set_xscale(XSCALE)
    ax.set_xlabel("number of datapoints")
    ax.set_ylabel("loss = -log probability")
    ax.set_title("Loss summary vs number of datapoints")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, "loss_summary_vs_n_train.png")
    return fig


def plot_loss_gap_and_drop():
    rows = finite_loss_rows()
    if not rows:
        return None
    x = np.array([row["n_train"] for row in rows], dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].plot(x, [row["final_loss_gap"] for row in rows], marker="o", linewidth=1.9)
    axes[0].axhline(0, color="black", linewidth=0.9, alpha=0.55)
    axes[0].set_title("Final validation-training loss gap")
    axes[0].set_ylabel("validation loss - training loss")

    axes[1].plot(x, [row["training_loss_drop"] for row in rows], marker="o", linewidth=1.9, label="training")
    axes[1].plot(x, [row["validation_loss_drop"] for row in rows], marker="o", linewidth=1.9, label="validation")
    axes[1].set_title("Loss drop from first to final epoch")
    axes[1].set_ylabel("first loss - final loss")
    axes[1].legend(frameon=False)

    for ax in axes:
        ax.set_xscale(XSCALE)
        ax.set_xlabel("number of datapoints")
        ax.grid(True, which="both", alpha=0.25)
    fig.tight_layout()
    save_figure(fig, "loss_gap_and_drop_vs_n_train.png")
    return fig


plot_loss_summary();
plot_loss_gap_and_drop();

## Numerical Tables

The full numeric summaries are saved as CSV files when `SAVE_TABLES = True`.

In [ ]:
print("Posterior metrics sorted by datapoints and parameter")
print_rows(
    sorted(posterior_rows, key=lambda row: (row["n_train"], PARAM_NAMES.index(row["parameter"]))),
    ["n_train", "parameter", "mean", "std", "std_over_prior_range", "variance_over_prior_range2", pull_metric],
)

print("\nLoss metrics sorted by datapoints")
print_rows(
    run_rows,
    ["n_train", "loss_available", "final_training_loss", "final_validation_loss", "best_validation_loss", "final_loss_gap"],
)